# 04 — Building a Text-to-Text Generation System with Transformers

## 📚 Learning Objectives

By completing this notebook, you will:
- Build an **encoder–decoder Transformer** (the T5/BART architecture family) in pure PyTorch
- Understand the seq2seq machinery: SOS/EOS tokens, teacher forcing, the causal target mask, greedy decoding
- Learn why Transformers need **positional encodings** — and see the model fail without enough signal about order
- **Evaluate** the system with exact-match accuracy on held-out inputs

## 🔗 Prerequisites

- ✅ Transformer architecture and attention (Course 07)
- ✅ Example 01 (autoregressive generation)

---

## Introduction

**Text-to-text** systems (T5, BART, translation models) frame every task as "text in → text out": an **encoder** reads the input sequence, a **decoder** generates the output autoregressively while attending to the encoder's states. We build a tiny one on a toy task chosen so success is unambiguous: **reverse a sequence of digits**. If the model truly learns the transformation, held-out accuracy should be ~100% — and we measure it.

**One detail is load-bearing: positional encodings.** Self-attention is *permutation-invariant* — without position information the model literally cannot represent "first digit" vs "last digit", so a task like reversal is unlearnable. Our model adds a learned positional embedding to every token; try removing `self.add_pos` after class and watch accuracy collapse.


In [1]:
# WHAT/WHY: build and train a small encoder-decoder Transformer that learns
# to REVERSE digit sequences, then measure exact-match accuracy on 200 unseen
# sequences. Note the positional embeddings — without them, attention cannot
# see token order and this task is unlearnable.
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import numpy as np
print(f'PyTorch {torch.__version__}')

# ── Toy task: reverse a sequence of digits ────────────────────────────────
SOS, EOS, PAD = 10, 11, 12          # special tokens appended to the digit vocab
V = 13                               # vocabulary: digits 0-9 + SOS/EOS/PAD
torch.manual_seed(0); np.random.seed(0)

def make_pair(n=6):
    s = np.random.randint(0, 10, n).tolist()
    return s, list(reversed(s))

def collate(pairs):
    # teacher forcing: decoder INPUT starts with SOS, decoder TARGET ends with EOS
    src     = torch.tensor([p[0] for p in pairs], dtype=torch.long)
    tgt_in  = torch.tensor([[SOS] + p[1] for p in pairs], dtype=torch.long)
    tgt_out = torch.tensor([p[1] + [EOS] for p in pairs], dtype=torch.long)
    return src, tgt_in, tgt_out

# ── Encoder-decoder Transformer with LEARNED positional embeddings ────────
class Seq2SeqTransformer(nn.Module):
    def __init__(self, V, d=32, nhead=4, nlayers=2, max_len=16):
        super().__init__()
        self.src_emb = nn.Embedding(V, d)
        self.tgt_emb = nn.Embedding(V, d)
        self.pos_emb = nn.Embedding(max_len, d)   # position i → learned vector
        self.transformer = nn.Transformer(d, nhead, nlayers, nlayers,
                                          dim_feedforward=64, batch_first=True)
        self.fc = nn.Linear(d, V)
    def add_pos(self, emb):
        # add position information — attention alone is order-blind
        pos = torch.arange(emb.size(1), device=emb.device)
        return emb + self.pos_emb(pos)[None, :, :]
    def forward(self, src, tgt):
        # causal mask: decoder position t may only attend to positions ≤ t
        mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1))
        out = self.transformer(self.add_pos(self.src_emb(src)),
                               self.add_pos(self.tgt_emb(tgt)),
                               tgt_mask=mask, tgt_is_causal=True)
        return self.fc(out)

model   = Seq2SeqTransformer(V)
opt     = optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)
pairs   = [make_pair() for _ in range(4000)]

# ── Training with teacher forcing ─────────────────────────────────────────
for epoch in range(20):
    model.train(); el = 0
    for i in range(0, len(pairs), 128):
        src, tgt_in, tgt_out = collate(pairs[i:i+128])
        logits = model(src, tgt_in)                     # (B, T, V)
        loss   = loss_fn(logits.reshape(-1, V), tgt_out.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step(); el += loss.item()
    if (epoch+1) % 5 == 0:
        print(f'Epoch {epoch+1}: summed batch loss={el:.4f}')

# ── Greedy decoding: generate the output one token at a time ──────────────
def decode(src_seq, max_len=10):
    model.eval()
    src = torch.tensor([src_seq], dtype=torch.long)
    tgt = torch.tensor([[SOS]], dtype=torch.long)      # start with SOS
    for _ in range(max_len):
        with torch.no_grad():
            logits = model(src, tgt)
        nxt = logits[0, -1].argmax().item()            # most likely next token
        if nxt == EOS:
            break                                       # model says it is done
        tgt = torch.cat([tgt, torch.tensor([[nxt]])], dim=1)
    return tgt[0, 1:].tolist()

# ── Evaluation: exact match on 200 UNSEEN sequences ───────────────────────
test_pairs = [make_pair() for _ in range(200)]
correct = sum(decode(s) == t for s, t in test_pairs)
print('\nSample predictions (input → expected | model):')
for s, t in test_pairs[:5]:
    pred = decode(s)
    print(f'  {s} → {t} | {pred}  {"✓" if pred == t else "✗"}')
print(f'\nExact-match accuracy on 200 unseen sequences: {correct}/200 = {correct/200:.1%}')


PyTorch 2.13.0


Epoch 5: summed batch loss=0.2446


Epoch 10: summed batch loss=0.0802


Epoch 15: summed batch loss=0.0874


Epoch 20: summed batch loss=0.0209



Sample predictions (input → expected | model):
  [8, 5, 0, 7, 5, 2] → [2, 5, 7, 0, 5, 8] | [2, 5, 7, 0, 5, 8]  ✓
  [0, 2, 1, 1, 9, 2] → [2, 9, 1, 1, 2, 0] | [2, 9, 1, 1, 2, 0]  ✓
  [1, 5, 5, 6, 9, 2] → [2, 9, 6, 5, 5, 1] | [2, 9, 6, 5, 5, 1]  ✓
  [3, 8, 2, 2, 4, 5] → [5, 4, 2, 2, 8, 3] | [5, 4, 2, 2, 8, 3]  ✓
  [3, 5, 2, 1, 0, 3] → [3, 0, 1, 2, 5, 3] | [3, 0, 1, 2, 5, 3]  ✓

Exact-match accuracy on 200 unseen sequences: 200/200 = 100.0%


## 🌍 Related Worked Example — Decoder-Only Generation

The seq2seq system above is an **encoder–decoder** (input text → output text). The other dominant design is **decoder-only** (GPT): no separate input encoder — the "input" is simply the start of the sequence and the model continues it. For contrast, revisit the decoder-only char-level generator in **example 01**: it has no separate encoder — it simply continues its input, GPT-style. The references below cover both designs.


## 📚 References & Further Reading

**Papers:**
- Vaswani et al. (2017) — [Attention Is All You Need](https://arxiv.org/abs/1706.03762) *(the Transformer; §3.5 explains positional encodings)*
- Raffel et al. (2020) — [T5: Exploring the Limits of Transfer Learning](https://arxiv.org/abs/1910.10683) *(text-to-text framing)*
- Lewis et al. (2020) — [BART](https://arxiv.org/abs/1910.13461)

**Docs:** [`nn.Transformer`](https://pytorch.org/docs/stable/generated/torch.nn.Transformer.html)


## 📝 Summary

In **04 — Building a Text-to-Text Generation System** you built an encoder–decoder Transformer with SOS/EOS handling, teacher forcing, a causal target mask, **learned positional embeddings**, and greedy decoding — and evaluated it honestly: check the printed exact-match accuracy on 200 unseen sequences. The positional embeddings are the piece to remember: without order information, attention cannot solve order-dependent tasks at all. T5/BART are this same architecture at scale.
